In [ ]:
# pip install git+https://github.com/huggingface/transformers
!pip install transformers==4.57.0 # currently, V4.57.0 is not released
!pip install -q accelerate bitsandbytes torch torchvision pillow
!pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.6 MB/s eta 0:00:00
Reason for being yanked: Error in the setup causing installation issues
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 28.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 89.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=253780426 sha256=4e2f9e39313266b1544b68138b15b91ee6221eccf14f7902b7c6620351340810
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

# default: Load the model on the available device(s)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-8B-Instruct", dtype="auto", device_map="auto"
)

model = model.eval()

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/2.72G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [ ]:
prompt = f"""

You are an embodied agent instruction generator.

Two images show the SAME environment:
- Image 1: Frontal (egocentric)
- Image 2: Top-down (bird’s-eye)

Generate exactly 3 tasks per level.

LEVEL_1: Navigation only
LEVEL_2: 1–2 object interactions
LEVEL_3: 3+ object interactions

Rules:
- Use ONLY visible objects
- Start/End must reference objects
- Do NOT use left/right/front/back/center

Format:

ENVIRONMENT SUMMARY:
<2–3 sentences + list of objects>

LEVEL_1 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: none
LEVEL_1 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: none
LEVEL_1 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: none
LEVEL_2 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: obj1, obj2
LEVEL_2 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: obj1, obj2
LEVEL_2 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: obj1, obj2
LEVEL_3 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: obj1, obj2, obj3
LEVEL_3 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: obj1, obj2, obj3
LEVEL_3 | TASK: ... | START: Near <object> | END: Near <object> | INTERACT: obj1, obj2, obj3
"""
images = [
    '/content/35_Bus loop with.4_1.png',
    '/content/35_Bus loop with.4_2.png'
]

In [ ]:
# Load images using PIL
pil_images = [Image.open(img_path) for img_path in images]

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": pil_images[0], # Use PIL Image object
                "name": "image1",
            },
            {
                "type": "image",
                "image": pil_images[1], # Use PIL Image object
                "name": "image2",
            },
            {"type": "text", "text": prompt},
        ],
    }
]

# Preparation for inference
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

In [ ]:
# Inference: Generation of the output
with torch.no_grad():
  generated_ids = model.generate(**inputs, max_new_tokens=512)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['ENVIRONMENT SUMMARY:\nThis is a miniature bus station scene with a blue-and-white bus parked on a dark asphalt lot marked with yellow lines. Key objects include the bus, a multi-story station building, a fuel pump station, and a small kiosk or ticket booth. The scene is viewed from two angles: a frontal perspective and a top-down view.\n\nLEVEL_1 | TASK: Walk from the fuel pump to the bus | START: Near fuel pump | END: Near bus | INTERACT: none\nLEVEL_1 | TASK: Walk from the bus to the station building | START: Near bus | END: Near station building | INTERACT: none\nLEVEL_1 | TASK: Walk from the kiosk to the station building | START: Near kiosk | END: Near station building | INTERACT: none\nLEVEL_2 | TASK: Move from the fuel pump to the bus while avoiding the yellow lines | START: Near fuel pump | END: Near bus | INTERACT: fuel pump, bus\nLEVEL_2 | TASK: Move from the kiosk to the station building while avoiding the yellow lines | START: Near kiosk | END: Near station building | INTE